# Test du rapprochement des gares

Ce notebook vérifie `preparer_gares` et `ajouter_uic_regularite`, et repère les gares à corriger.

Les corrections se font **dans les fonctions** (fichier `preprocessing.py`), puis on relance les cellules.

In [1]:
from difflib import get_close_matches

import pandas as pd

from railinsight.data.loader import GARES, REGULARITE, load
from railinsight.data.preprocessing import preprocess_regularite, preprocess_gares
from railinsight.data.quality import normaliser_noms

## 1. Lancer les fonctions

In [2]:
df_gares_brut = load(GARES)
df_regularite_brut = load(REGULARITE)

df_gares_propre = preprocess_gares(df_gares_brut)
df_regularite = preprocess_regularite(df_regularite_brut, df_gares_propre)

0 gares sans code UIC : []


## 2. Vérifications

In [3]:
print("Gares :", df_gares_brut.shape, "->", df_gares_propre.shape)
print("Codes UIC en double :", df_gares_propre["code_uic"].duplicated().sum(), "(doit valoir 0)")
print("Noms de gares en double :", df_gares_propre["gare"].duplicated().sum(), "(idéalement 0)")

complet = df_regularite[["uic_depart", "uic_arrivee"]].notna().all(axis=1)
print(f"Lignes de régularité avec les deux codes UIC : {complet.mean():.1%}")

Gares : (6469, 18) -> (6065, 7)
Codes UIC en double : 0 (doit valoir 0)
Noms de gares en double : 4 (idéalement 0)
Lignes de régularité avec les deux codes UIC : 100.0%


In [4]:
doublons = df_gares_propre[
    df_gares_propre["gare"].duplicated(keep=False)
].sort_values("gare")

print(f"Nombre de lignes concernées : {len(doublons)}")

display(doublons[["code_uic", "libelle", "gare", "commune", "departement"]])

Nombre de lignes concernées : 8


,code_uic,libelle,gare,commune,departement
1616,87182410,Cernay,CERNAY,CERNAY,HAUT-RHIN
4516,87276063,Cernay,CERNAY,ERMONT,VAL-D'OISE
1996,87746842,La Joux,LA JOUX,CHAMONIX-MONT-BLANC,HAUTE-SAVOIE
2235,87715110,La Joux,LA JOUX,SUPT,JURA
2773,87174268,Pringy,PRINGY,PRINGY,MARNE
4170,87746206,Pringy,PRINGY,ANNECY,HAUTE-SAVOIE
5011,87272021,Villaines,VILLAINES,VILLAINES-SOUS-BOIS,VAL-D'OISE
5507,87396457,Villaines,VILLAINES,VILLAINES-SOUS-MALICORNE,SARTHE


## 3. Gares sans code UIC

`plus_proche` est la gare de la liste la plus ressemblante.

- **Suggestion correcte** : ajoutez la gare dans `uic_manuels` (fonction `ajouter_uic_regularite`), avec le `code_uic` suggéré.
- **Aucune suggestion pertinente** : la gare est absente de la liste. Ajoutez-la dans `gares_a_ajouter` (fonction `preparer_gares`), avec son vrai code UIC et ses coordonnées.

In [5]:
manquantes = pd.concat([
    df_regularite.loc[df_regularite["uic_depart"].isna(), "gare_depart"],
    df_regularite.loc[df_regularite["uic_arrivee"].isna(), "gare_arrivee"],
]).drop_duplicates().sort_values()

noms_connus = df_gares_propre["gare"].dropna().tolist()


def plus_proche(nom: str):
    resultats = get_close_matches(nom, noms_connus, n=1, cutoff=0.5)
    return resultats[0] if resultats else None


a_corriger = pd.DataFrame({"gare_regularite": manquantes.values})
a_corriger["plus_proche"] = normaliser_noms(a_corriger["gare_regularite"]).apply(plus_proche)

gares_ref = df_gares_propre.drop_duplicates(subset="gare")[["gare", "libelle", "code_uic"]]
a_corriger = a_corriger.merge(gares_ref, left_on="plus_proche", right_on="gare", how="left")
a_corriger = a_corriger.drop(columns="gare").rename(
    columns={"libelle": "libelle_suggere", "code_uic": "code_uic_suggere"}
)
a_corriger["code_uic_suggere"] = a_corriger["code_uic_suggere"].astype("Int64")  # sans décimale
a_corriger

,gare_regularite,plus_proche,libelle_suggere,code_uic_suggere


## 4. Lignes à coller dans `uic_manuels`

Gardez seulement celles dont la suggestion est correcte, et remplacez le `0` par le code UIC.

In [6]:
for nom in a_corriger["gare_regularite"]:
    print(f'        "{nom}": 0,')